In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when

# --- CONFIGURAÇÃO BLINDADA DO SPARK ---
os.environ["AWS_REGION"] = "us-east-1"
os.environ["AWS_ACCESS_KEY_ID"] = "admin"
os.environ["AWS_SECRET_ACCESS_KEY"] = "password"

packages = [
    "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.3",
    "org.apache.iceberg:iceberg-aws-bundle:1.4.3",
    "org.apache.hadoop:hadoop-aws:3.3.4",
    "com.amazonaws:aws-java-sdk-bundle:1.12.262"
]

spark = SparkSession.builder \
    .appName("Modelagem-Ouro-Cruzamento-CEIS") \
    .config("spark.jars.packages", ",".join(packages)) \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.iceberg", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.iceberg.type", "hive") \
    .config("spark.sql.catalog.iceberg.uri", "thrift://hive-metastore:9083") \
    .config("spark.sql.catalog.iceberg.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.iceberg.warehouse", "s3a://warehouse/") \
    .config("spark.sql.catalog.iceberg.s3.endpoint", "http://minio:9000") \
    .config("spark.sql.catalog.iceberg.client.region", "us-east-1") \
    .config("spark.sql.catalog.iceberg.s3.path-style-access", "true") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
spark.sql("CREATE NAMESPACE IF NOT EXISTS iceberg.gold")

print("Carregando tabelas da camada Prata...")

df_cnpj = spark.table("iceberg.silver.cnpj_cleansed") 

df_ceis = spark.table("iceberg.silver.ceis_cleansed") \
    .drop("tipo_pessoa", "data_ingestao")

print("Executando cruzamento pelo CNPJ...")

df_cruzamento = df_cnpj.join(
    df_ceis,
    df_cnpj["cnpj"] == df_ceis["cpf_cnpj"],
    how="left"
)

print("Aplicando flag de auditoria...")

df_gold = df_cruzamento.withColumn(
    "alerta_sancao", 
    when(col("categoria_sancao").isNotNull(), "SIM").otherwise("NAO")
)

print("Gravando camada Ouro...")
df_gold.writeTo("iceberg.gold.base_auditada_ceis") \
    .tableProperty("format-version", "2") \
    .using("iceberg") \
    .createOrReplace()

print("Sucesso! Cruzamento finalizado de forma agnóstica.")

:: loading settings :: url = jar:file:/usr/local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/vscode/.ivy2/cache
The jars for the packages stored in: /home/vscode/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-39f51c3a-16b3-4e06-8a70-3f676344b184;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.4.3 in central
	found org.apache.iceberg#iceberg-aws-bundle;1.4.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 210ms :: artifacts dl 13ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [defaul

Carregando tabelas da camada Prata...
Executando cruzamento pelo CNPJ...
Aplicando flag de auditoria...
Gravando camada Ouro...


26/04/20 13:24:29 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Sucesso! Cruzamento finalizado de forma agnóstica.


In [ ]:
df_show = spark.table("iceberg.gold.base_auditada_ceis").where(col("alerta_sancao") == "SIM")
df_show.show(truncate=False)